# B7 U10P Finalize-Only
Dedicated recovery notebook. Reads only completed source jobs; no M1 load, evaluator tests, smoke or selection. Python patch differences only; NumPy/pandas exact. Set both SHA pins to the Chat-reviewed Producer Implementation Freeze.

In [ ]:
from pathlib import Path
import os, sys, subprocess
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
REVIEWED_SHA = ''  # Paste Chat-reviewed U10P Implementation Freeze SHA, never branch HEAD.
REVIEWED_PRODUCER_SHA = ''  # Finalize: explicitly pin the same reviewed Producer Freeze.
REPO = Path('/content/b7_u10p_repo')
DATA_ROOT = Path('/content/drive/MyDrive/REPLACE_WITH_FROZEN_DATA_ROOT')
LOCAL = Path('/content/b7_u10p_run')
DRIVE = Path('/content/drive/MyDrive/REPLACE_WITH_NEW_U10P_CHECKPOINT_ROOT')
FINAL_OUTPUT = Path('/content/b7_u10p_finalize')  # Must not exist.
ARCHIVE = Path('/content/drive/MyDrive/REPLACE_WITH_NEW_U10P_ARCHIVE_ROOT')
APPROVAL = ''  # Separate explicit Chat authorization is required.
RUN_MOUNT = False
RUN_INSTALL = False
RUN_CHECKOUT = False
RUN_VERIFY_FREEZE = False
RUN_FINALIZE_ONLY = False
RUN_ARCHIVE = False
RUN_CALENDAR_AUDIT = False


In [ ]:
if RUN_MOUNT:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
if RUN_INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy==2.3.5', 'pandas==2.2.3'], check=True)
    print('Restart runtime if these packages were already imported; then rerun configuration.')


In [ ]:
if RUN_CHECKOUT:
    import re
    assert re.fullmatch('[0-9a-f]{40}', REVIEWED_SHA)
    if not REPO.exists():
        subprocess.run(['git','clone','https://github.com/TR-KJ/time-entry-portfolio-lab.git',str(REPO)],check=True)
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()
    subprocess.run(['git','-C',str(REPO),'fetch','origin',REVIEWED_SHA],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',REVIEWED_SHA],check=True)
    assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==REVIEWED_SHA
    sys.path.insert(0,str(REPO/'src/research'))


In [ ]:
if RUN_VERIFY_FREEZE:
    from b7.u10p_finalize_only import current_preflight
    print(current_preflight(REVIEWED_SHA,REVIEWED_PRODUCER_SHA))


In [ ]:
if RUN_FINALIZE_ONLY:
    from b7.u10p_finalize_only import finalize_from_completed_jobs
    review=finalize_from_completed_jobs(DRIVE,FINAL_OUTPUT,REVIEWED_SHA,REVIEWED_PRODUCER_SHA,approval=APPROVAL,progress=print)
    print(review)


In [ ]:
if RUN_ARCHIVE:
    from b7.u10p_runtime import archive_completed
    archive_completed(FINAL_OUTPUT,ARCHIVE)


## Frozen event calendar identity audit


In [ ]:
if RUN_CALENDAR_AUDIT:
    from b7.u10_calendar import audit_calendar
    c = audit_calendar()
    print({'Status':'PASS','Events':len(c['Events']),'CalendarSourceCommit':c['CalendarSourceCommit']})
